# Heart Disease Prediction Project

This notebook is structured in phases:

1. **Phase 1: Exploratory Data Analysis (EDA)**
2. **Phase 2: Data Preprocessing**
3. **Phase 3: Logistic Regression Model**
4. **Phase 4: K-Nearest Neighbors (KNN) Model**
5. **Phase 5: Support Vector Machine (SVM) Model**
6. **Phase 6: Model Comparison and ROC Curves**


## Phase 0: Import Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

import warnings
warnings.filterwarnings("ignore")


## Phase 1: Exploratory Data Analysis (EDA)

In this phase, we only understand the dataset. We do not train models here.


In [ ]:
# Load dataset
df = pd.read_csv("heart_disease_dataset.csv")

# Display first five rows
df.head()


In [ ]:
# Dataset shape
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


In [ ]:
# Basic information about columns and data types
df.info()


In [ ]:
# Statistical summary for numerical columns
df.describe()


In [ ]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print(missing_values)


In [ ]:
# Target column distribution
print(df["heart_disease"].value_counts(dropna=False))

sns.countplot(x="heart_disease", data=df)
plt.title("Target Distribution: 0 = No Disease, 1 = Disease")
plt.xlabel("Heart Disease")
plt.ylabel("Count")
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
# Correlation of features with target
corr = df.corr(numeric_only=True)
target_corr = corr["heart_disease"].abs().sort_values(ascending=False)

print(target_corr)

important_cols = target_corr.index[1:6]
print("Top correlated columns:", list(important_cols))


In [ ]:
# Boxplots of top correlated numerical columns before preprocessing
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[list(important_cols)])
plt.title("Boxplot of Important Columns Before Preprocessing")
plt.xticks(rotation=45)
plt.show()


## Phase 2: Data Preprocessing

In this phase, we clean the data, split features/target, select features, and scale data.


In [ ]:
# Create a copy for preprocessing
df_clean = df.copy()


In [ ]:
# Remove duplicate rows if present
df_clean = df_clean.drop_duplicates()
print("Shape after duplicate removal:", df_clean.shape)


In [ ]:
# Fill missing values
# Numerical columns: median
# Categorical columns: mode

numeric_cols = df_clean.select_dtypes(include=np.number).columns
categorical_cols = df_clean.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(df_clean.isnull().sum())


In [ ]:
# Remove outliers using IQR method for numerical columns except target
num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

if "heart_disease" in num_cols:
    num_cols.remove("heart_disease")

for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]

print("Shape after outlier removal:", df_clean.shape)


In [ ]:
# Boxplots after outlier removal
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_clean[list(important_cols)])
plt.title("Boxplot of Important Columns After Outlier Removal")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Separate features and target
X = df_clean.drop("heart_disease", axis=1)
y = df_clean["heart_disease"]   # Keep y as a Series, not a DataFrame

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Convert categorical columns into numerical form if any exist
X = pd.get_dummies(X, drop_first=True)
X.head()


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


In [ ]:
# Feature selection using mutual information
k_value = min(8, X_train.shape[1])
selector = SelectKBest(score_func=mutual_info_classif, k=k_value)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_cols = X_train.columns[selector.get_support()]
print("Selected Features:", list(selected_cols))

X_train_selected = pd.DataFrame(X_train_selected, columns=selected_cols, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected, columns=selected_cols, index=X_test.index)


In [ ]:
# Standard scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=selected_cols, index=X_train_selected.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=selected_cols, index=X_test_selected.index)

X_train_scaled.head()


## Common Evaluation Function


In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n===== {model_name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, zero_division=0))
    print("F1 Score:", f1_score(y_true, y_pred, zero_division=0))
    
    print("\nClassification Report:\n", classification_report(y_true, y_pred, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


## Phase 3: Logistic Regression Model


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
evaluate_model(y_test, y_pred_lr, "Logistic Regression")


## Phase 4: K-Nearest Neighbors Model


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
evaluate_model(y_test, y_pred_knn, "KNN")


## Phase 5: Support Vector Machine Model


In [ ]:
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
evaluate_model(y_test, y_pred_svm, "SVM")


## Phase 6: Model Comparison


In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_svm)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_svm, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_knn, zero_division=0),
        recall_score(y_test, y_pred_svm, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_knn, zero_division=0),
        f1_score(y_test, y_pred_svm, zero_division=0)
    ]
})

results


In [ ]:
# Bar chart comparison
results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(kind="bar", figsize=(10, 6))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.show()


In [ ]:
# ROC curve comparison
models = {
    "Logistic Regression": lr,
    "KNN": knn,
    "SVM": svm
}

plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} AUC = {roc_auc:.2f}")

plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()
